# Feasibility Setup

In [1]:
import datetime as dt
import json
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs


from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.by import By

from IPython.display import Image, display

# Project A: Feasibility Testing
Testing the feasibility of Project A - the Swiss ski day-pass pricing

In [2]:
RAW = Path.cwd().parent / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)

# Each resort: its ticket shop URL and the canton whose school holidays apply.
RESORTS = {
    "Laax":   {"url": "https://shop.laax.com/en/home",                 "canton": "GR"},
    "Titlis": {"url": "https://shop.titlis.ch/en/products/skitickets", "canton": "OW"},
    # add more as you verify them, one line each
}

SEASON = ("2026-11-01", "2027-04-30")
VALIDATION_RESORT = "Laax"          # the one we also scrape via the DOM, to cross-check

API = "https://api.pricenow.dev/api/booking/products/{pid}/journey/validities"

UA = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
      "(KHTML, like Gecko) Chrome/154.0.0.0 Safari/537.36 "
      "CIP-HS26-208-HSLU-student-project (adam.ogle@stud.hslu.ch)")

CELL_TXT = re.compile(r"^(\d{1,2})\s+(\d{1,3}[.,]\d{2})$")
DOM_COLS = ["date", "price_chf", "raw_text", "classes", "disabled"]

FORM_STATE = {
    "attributes": {"duration": "1d"},
    "allocation": {"kind": "persontype",
                   "values": [{"kind": "anonymous", "personType": "adult", "quantity": 1}]},
}

# PLACEHOLDER — replace with the official cantonal school-holiday calendars
HOLIDAY_WINDOWS = {
    "GR": {"christmas": ("2026-12-19", "2027-01-03"),
           "sport":     ("2027-02-06", "2027-02-21"),
           "easter":    ("2027-03-26", "2027-04-11")},
    "OW": {"christmas": ("2026-12-19", "2027-01-03"),
           "sport":     ("2027-02-13", "2027-02-21"),
           "easter":    ("2027-03-26", "2027-04-11")},
}


# ---------- browser ----------
def make_driver(headless=False, perf_log=True):
    opts = Options()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--window-size=1500,1100")
    opts.add_argument(f"--user-agent={UA}")
    if perf_log:
        opts.set_capability("goog:loggingPrefs", {"performance": "ALL"})
    return webdriver.Chrome(options=opts)


def robust_click(d, el):
    d.execute_script("arguments[0].scrollIntoView({block:'center'});", el)
    time.sleep(0.4)
    for how in ("native", "actions", "js"):
        try:
            if how == "native":
                el.click()
            elif how == "actions":
                ActionChains(d).move_to_element(el).pause(0.2).click().perform()
            else:
                d.execute_script("arguments[0].click();", el)
            return how
        except Exception:
            continue
    raise RuntimeError("click failed")


def _clickables(d):
    return d.find_elements(By.TAG_NAME, "button") + d.find_elements(By.TAG_NAME, "a")


def click_text(d, text):
    for el in _clickables(d):
        if " ".join((el.text or "").split()) == text:
            return robust_click(d, el)
    raise RuntimeError(f"not found: {text!r}")


def click_regex(d, pattern):
    rx = re.compile(pattern)
    for el in _clickables(d):
        t = " ".join((el.text or "").split())
        if rx.search(t):
            robust_click(d, el)
            return t
    raise RuntimeError(f"no match /{pattern}/")


def dismiss_cookies(d):
    """Must run BEFORE the calendar modal opens — dismissing it after closes the modal."""
    for lbl in ("Allow all", "Accept", "OK", "Alle zulassen", "Tout accepter"):
        try:
            click_text(d, lbl)
            return True
        except Exception:
            pass
    for f in d.find_elements(By.TAG_NAME, "iframe"):
        if "cookiebot" in (f.get_attribute("src") or ""):
            d.switch_to.frame(f)
            for lbl in ("Allow all", "OK", "Accept", "Alle zulassen"):
                try:
                    click_text(d, lbl)
                    d.switch_to.default_content()
                    return True
                except Exception:
                    pass
            d.switch_to.default_content()
    return False


# ---------- DOM extraction (validation path) ----------
def parse_calendar(html):
    """One row per real calendar day. Padding cells from adjacent months are dropped."""
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for td in soup.select("td[data-day]"):
        classes = td.get("class", [])
        if "calendar__outside" in classes or "calendar__hidden" in classes:
            continue
        txt = " ".join(td.get_text(" ", strip=True).split())
        m = CELL_TXT.match(txt)
        rows.append({
            "date": td["data-day"],
            "price_chf": float(m.group(2).replace(",", ".")) if m else None,
            "raw_text": txt,
            "classes": " ".join(classes),
            "disabled": "calendar__disabled" in classes,
        })
    return pd.DataFrame(rows, columns=DOM_COLS)


def find_next_arrow(d):
    for b in d.find_elements(By.TAG_NAME, "button"):
        lab = ((b.get_attribute("aria-label") or "") + " " + (b.get_attribute("class") or "")).lower()
        if any(k in lab for k in ("next", "forward", "chevron-right", "arrow-right")):
            disabled = (
                b.get_attribute("disabled") is not None
                or (b.get_attribute("aria-disabled") or "").lower() == "true"
                or "disabled" in (b.get_attribute("class") or "").lower()
            )
            return b, disabled
    return None, None


def crawl_dom_calendar(d, name, stamp, max_steps=14, verbose=True):
    chunks, seen, dry = [], set(), 0
    for step in range(max_steps):
        (RAW / f"{name.lower()}_{stamp}_step{step}.html").write_text(
            d.page_source, encoding="utf-8")
        df = parse_calendar(d.page_source)
        new = df[~df["date"].isin(seen)]
        n_priced = int(new["price_chf"].notna().sum()) if len(new) else 0
        if verbose:
            msg = f"    step {step}: {len(new)} new, {n_priced} priced"
            if len(new):
                msg += f", {new['date'].min()} → {new['date'].max()}"
            print(msg)
        if len(new):
            chunks.append(new)
            seen.update(new["date"])
        dry = dry + 1 if n_priced == 0 else 0
        if dry >= 2:
            break
        nxt, disabled = find_next_arrow(d)
        if nxt is None or disabled:
            break
        try:
            robust_click(d, nxt)
        except Exception:
            time.sleep(2)
            nxt, _ = find_next_arrow(d)
            robust_click(d, nxt)
        time.sleep(2.5)

    if not chunks:
        return pd.DataFrame(columns=DOM_COLS)
    out = pd.concat(chunks, ignore_index=True).drop_duplicates("date")
    out["date"] = pd.to_datetime(out["date"])
    out["price_chf"] = pd.to_numeric(out["price_chf"], errors="coerce")
    return out.sort_values("date").reset_index(drop=True)


# ---------- API ----------
def capture_api_credentials(d):
    """Product ID, headers and the page's own formState, from the GET it already made."""
    for entry in d.get_log("performance"):
        msg = json.loads(entry["message"])["message"]
        if msg.get("method") != "Network.requestWillBeSent":
            continue
        req = msg["params"]["request"]
        if "journey/validities" not in req["url"]:
            continue
        if (req.get("method") or "").upper() != "GET":
            continue
        pid = re.search(r"/products/(\d+)/", req["url"])
        qs = parse_qs(urlparse(req["url"]).query)
        form_state = qs.get("formState", [None])[0]
        hdrs = {k: v for k, v in req["headers"].items()
                if k.lower() not in ("host", "content-length", "cookie", "accept-encoding")}
        return (pid.group(1) if pid else None), hdrs, form_state
    return None, None, None


def fetch_validities(product_id, headers, form_state, start, end, timeout=30):
    """form_state: the JSON string captured from that shop's own request."""
    params = {
        "from": f"{start}T00:00:00.000+01:00",
        "to":   f"{end}T23:59:59.999+01:00",
        "formState": form_state,
    }
    return requests.get(API.format(pid=product_id), params=params,
                        headers=headers, timeout=timeout)


def parse_validities(payload, resort, observed_at):
    rows = []
    for o in payload["json"]["data"]["options"]:
        frm = pd.to_datetime(o["value"]["from"], utc=True).tz_convert("Europe/Zurich")
        to  = pd.to_datetime(o["value"]["to"],   utc=True).tz_convert("Europe/Zurich")
        price = o.get("price") or {}
        mn, mx = price.get("minPrice"), price.get("maxPrice")
        rows.append({
            "resort": resort,
            "date": frm.normalize().tz_localize(None),
            "duration_days": int(round((to - frm).total_seconds() / 86400)),
            "label": o.get("label"),
            "is_available": o.get("isAvailable"),
            "capacity_kind": (o.get("availableCapacity") or {}).get("kind"),
            "price_chf": None if mn is None else mn / 100,
            "max_price_chf": None if mx is None else mx / 100,
            "observed_at": observed_at,
        })
    return pd.DataFrame(rows)


# ---------- one browser pass per shop ----------
def probe_shop(name, url, stamp, crawl_dom=False, headless=False, verbose=True):
    """Open a shop, confirm PriceNow, capture product ID + headers + formState,
    optionally crawl the DOM. Handles inline and modal calendar layouts."""
    out = {"resort": name, "url": url, "is_pricenow": False, "product_id": None,
           "headers": None, "form_state": None, "n_cells": 0, "layout": None,
           "dom": None, "error": None}
    d = make_driver(headless=headless, perf_log=True)
    try:
        d.get(url)
        time.sleep(8)
        for _ in range(5):
            if dismiss_cookies(d):
                break
            time.sleep(3)
        time.sleep(2)

        if d.find_elements(By.CSS_SELECTOR, "td[data-day]"):
            out["layout"] = "inline"
        else:
            try:
                click_regex(d, r"\d{2}\.\d{2}\.\d{4}")
                time.sleep(5)
                out["layout"] = "modal"
            except Exception as e:
                out["error"] = f"no calendar and no date field ({type(e).__name__})"
                d.save_screenshot(str(RAW / f"{name.lower()}_{stamp}_nocalendar.png"))

        if out["layout"]:
            out["n_cells"] = len(d.find_elements(By.CSS_SELECTOR, "td[data-day]"))
            d.save_screenshot(str(RAW / f"{name.lower()}_{stamp}_calendar.png"))

            pid, hdrs, form_state = capture_api_credentials(d)
            out["product_id"], out["headers"], out["form_state"] = pid, hdrs, form_state
            out["is_pricenow"] = pid is not None
            if not out["is_pricenow"]:
                out["error"] = "no PriceNow validities GET seen — different platform?"

            if crawl_dom and out["n_cells"]:
                if verbose:
                    print(f"  crawling DOM calendar for {name}:")
                out["dom"] = crawl_dom_calendar(d, name, stamp, verbose=verbose)
    except Exception as e:
        import traceback
        traceback.print_exc()
        out["error"] = f"{type(e).__name__}: {e}"
    finally:
        try:
            d.quit()
        except Exception:
            pass

    if verbose:
        dom_n = 0 if out["dom"] is None else int(out["dom"]["price_chf"].notna().sum())
        print(f"{name:10} pricenow={out['is_pricenow']}  product={out['product_id']}  "
              f"layout={out['layout']}  cells={out['n_cells']}  dom_priced={dom_n}  "
              f"{out['error'] or ''}")
    return out

In [3]:
OBSERVED_AT = dt.datetime.now().astimezone()
STAMP = f"{OBSERVED_AT:%Y%m%d_%H%M}"

probes = {}
for name, meta in RESORTS.items():
    probes[name] = probe_shop(name, meta["url"], STAMP,
                              crawl_dom=(name == VALIDATION_RESORT))

print("\nshops on PriceNow:", [n for n, p in probes.items() if p["is_pricenow"]])
print("failed:           ", [n for n, p in probes.items() if not p["is_pricenow"]])

  crawling DOM calendar for Laax:
    step 0: 61 new, 29 priced, 2026-11-01 → 2026-12-31
    step 1: 31 new, 31 priced, 2027-01-01 → 2027-01-31
    step 2: 28 new, 28 priced, 2027-02-01 → 2027-02-28
    step 3: 31 new, 31 priced, 2027-03-01 → 2027-03-31
    step 4: 30 new, 11 priced, 2027-04-01 → 2027-04-30
    step 5: 31 new, 0 priced, 2027-05-01 → 2027-05-31
    step 6: 30 new, 0 priced, 2027-06-01 → 2027-06-30
Laax       pricenow=True  product=1122216  layout=modal  cells=84  dom_priced=130  
Titlis     pricenow=True  product=180000  layout=inline  cells=42  dom_priced=0  

shops on PriceNow: ['Laax', 'Titlis']
failed:            []


In [4]:
frames = []
for name, p in probes.items():
    if not p["is_pricenow"]:
        print(f"{name:10} skipped — {p.get('error')}")
        continue
    print(f"\n{name} formState: {p['form_state']}")
    r = fetch_validities(p["product_id"], p["headers"], p["form_state"], *SEASON)
    print(f"{name:10} HTTP {r.status_code}  {len(r.content):>8} bytes")
    if not r.ok:
        print("   ", r.text[:200])
        continue
    (RAW / f"{name.lower()}_{STAMP}_validities.json").write_text(
        json.dumps(r.json()), encoding="utf-8")
    frames.append(parse_validities(r.json(), name, OBSERVED_AT.isoformat(timespec="seconds")))

if not frames:
    raise RuntimeError("no resort returned data — probe errors: "
                       + str({n: p.get("error") for n, p in probes.items()}))

allres = pd.concat(frames, ignore_index=True)
day = (allres[allres["duration_days"] == 1]
       .sort_values(["resort", "date"]).reset_index(drop=True))

print("\n--- 1-day passes by resort ---")
print(day.groupby("resort").agg(
    n=("price_chf", "count"), first=("date", "min"), last=("date", "max"),
    mean=("price_chf", "mean"), min=("price_chf", "min"), max=("price_chf", "max"),
).round(2))
print("\nproduct labels seen:", day["label"].unique())


Laax formState: {"attributes":{"duration":"1d"},"allocation":{"kind":"persontype","values":[{"kind":"anonymous","personType":"adult","quantity":1}]}}
Laax       HTTP 200    348220 bytes

Titlis formState: {"attributes":{"duration":"1d"},"allocation":{"kind":"persontype","values":[{"kind":"anonymous","personType":"adult_age20to999","quantity":1}]}}
Titlis     HTTP 200    412611 bytes

--- 1-day passes by resort ---
          n      first       last   mean   min   max
resort                                              
Laax    130 2026-11-28 2027-04-11  74.14  52.5  97.0
Titlis  152 2026-12-01 2027-05-01  57.95  51.0  66.5

product labels seen: <StringArray>
['1 Tag/Day', '1 Tag und 1/2 Tag ab 12:00 Uhr']
Length: 2, dtype: str


C:\Users\Adam\AppData\Local\Temp\ipykernel_4100\3754669469.py:28: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ).round(2))


In [5]:
dom = probes[VALIDATION_RESORT]["dom"]
dom_priced = dom[dom["price_chf"].notna()] if dom is not None else pd.DataFrame()

if len(dom_priced):
    api_one = day[day["resort"] == VALIDATION_RESORT][["date", "price_chf"]]
    chk = api_one.merge(dom_priced[["date", "price_chf"]], on="date",
                        how="outer", suffixes=("_api", "_dom"))
    chk["diff"] = chk["price_chf_api"] - chk["price_chf_dom"]

    print(f"validation resort: {VALIDATION_RESORT}")
    print("matched dates:  ", int(chk["diff"].notna().sum()))
    print("mismatches:     ", int((chk["diff"].abs() > 0.001).sum()))
    print("max abs diff:   ", chk["diff"].abs().max())
    print("API-only dates: ", int(chk["price_chf_dom"].isna().sum()))
    print("DOM-only dates: ", int(chk["price_chf_api"].isna().sum()))

    bad = chk[chk["diff"].abs() > 0.001]
    if len(bad):
        print("\nmismatching rows:")
        print(bad.to_string(index=False))
        print("\nKnown cause: the cell adjacent to the pre-selected date shows the marginal")
        print("'additional day' price, not the standalone day price. The API is correct.")
else:
    print("no DOM crawl available for cross-check")

validation resort: Laax
matched dates:   130
mismatches:      1
max abs diff:    10.0
API-only dates:  0
DOM-only dates:  0

mismatching rows:
      date  price_chf_api  price_chf_dom  diff
2026-11-29           54.0           44.0  10.0

Known cause: the cell adjacent to the pre-selected date shows the marginal
'additional day' price, not the standalone day price. The API is correct.


In [6]:
day = day.copy()
day["canton"] = day["resort"].map(lambda r: RESORTS[r]["canton"])
day["weekday"] = day["date"].dt.day_name()
day["is_weekend"] = day["date"].dt.dayofweek >= 5
day["days_until"] = (day["date"] - pd.Timestamp(OBSERVED_AT.date())).dt.days

day["holiday"] = None
for canton, windows in HOLIDAY_WINDOWS.items():
    for label, (a, b) in windows.items():
        mask = (day["canton"] == canton) & (day["date"] >= a) & (day["date"] <= b)
        day.loc[mask, "holiday"] = label
day["is_holiday"] = day["holiday"].notna()

out = RAW / f"prices_{STAMP}.csv"
day.to_csv(out, index=False)
print("saved:", out.name, "|", len(day), "rows,", day['resort'].nunique(), "resorts")

day.head(10)

saved: prices_20261004_1803.csv | 282 rows, 2 resorts


,resort,date,duration_days,label,is_available,capacity_kind,price_chf,max_price_chf,observed_at,canton,weekday,is_weekend,days_until,holiday,is_holiday
0,Laax,2026-11-28,1,1 Tag/Day,True,unlimited,54.5,54.5,2026-10-04T18:03:12+02:00,GR,Saturday,True,55,None,False
1,Laax,2026-11-29,1,1 Tag/Day,True,unlimited,54.0,54.0,2026-10-04T18:03:12+02:00,GR,Sunday,True,56,None,False
2,Laax,2026-12-05,1,1 Tag/Day,True,unlimited,56.0,56.0,2026-10-04T18:03:12+02:00,GR,Saturday,True,62,None,False
3,Laax,2026-12-06,1,1 Tag/Day,True,unlimited,55.0,55.0,2026-10-04T18:03:12+02:00,GR,Sunday,True,63,None,False
4,Laax,2026-12-07,1,1 Tag/Day,True,unlimited,52.5,52.5,2026-10-04T18:03:12+02:00,GR,Monday,False,64,None,False
5,Laax,2026-12-08,1,1 Tag/Day,True,unlimited,54.0,54.0,2026-10-04T18:03:12+02:00,GR,Tuesday,False,65,None,False
6,Laax,2026-12-09,1,1 Tag/Day,True,unlimited,53.5,53.5,2026-10-04T18:03:12+02:00,GR,Wednesday,False,66,None,False
7,Laax,2026-12-10,1,1 Tag/Day,True,unlimited,52.5,52.5,2026-10-04T18:03:12+02:00,GR,Thursday,False,67,None,False
8,Laax,2026-12-11,1,1 Tag/Day,True,unlimited,52.5,52.5,2026-10-04T18:03:12+02:00,GR,Friday,False,68,None,False
9,Laax,2026-12-12,1,1 Tag/Day,True,unlimited,62.5,62.5,2026-10-04T18:03:12+02:00,GR,Saturday,True,69,None,False


In [12]:
def boot_diff_ci(a, b, n=5000, seed=0):
    rng = np.random.default_rng(seed)
    a, b = np.asarray(a, float), np.asarray(b, float)
    if len(a) < 2 or len(b) < 2:
        return np.nan, (np.nan, np.nan)
    diffs = [rng.choice(a, a.size, True).mean() - rng.choice(b, b.size, True).mean()
             for _ in range(n)]
    return float(a.mean() - b.mean()), np.percentile(diffs, [2.5, 97.5])


for label, col in (("weekend", "is_weekend"), ("holiday", "is_holiday")):
    print(f"\n=== {label} premium ===")
    for resort, g in day.groupby("resort"):
        hi = g.loc[g[col], "price_chf"].dropna()
        lo = g.loc[~g[col], "price_chf"].dropna()
        d, ci = boot_diff_ci(hi, lo)
        if np.isnan(d):
            print(f"{resort:10} insufficient data")
            continue
        print(f"{resort:10} CHF {d:+6.2f}  95% CI [{ci[0]:+6.2f}, {ci[1]:+6.2f}]  "
              f"({d / lo.mean() * 100:+5.1f}%)   n={len(hi)}/{len(lo)}")

print("\nmean price by resort and holiday window:")
print(day.pivot_table(index="holiday", columns="resort", values="price_chf",
                      aggfunc="mean", dropna=False).round(2))


print("=== holiday premium, WEEKDAYS ONLY ===")
wd_only = day[~day["is_weekend"]]
for resort, g in wd_only.groupby("resort"):
    hi = g.loc[g["is_holiday"], "price_chf"].dropna()
    lo = g.loc[~g["is_holiday"], "price_chf"].dropna()
    d, ci = boot_diff_ci(hi, lo)
    print(f"{resort:10} CHF {d:+6.2f}  95% CI [{ci[0]:+6.2f}, {ci[1]:+6.2f}]  "
          f"({d / lo.mean() * 100:+5.1f}%)   n={len(hi)}/{len(lo)}")

print("\n=== holiday premium, WEEKENDS ONLY ===")
we_only = day[day["is_weekend"]]
for resort, g in we_only.groupby("resort"):
    hi = g.loc[g["is_holiday"], "price_chf"].dropna()
    lo = g.loc[~g["is_holiday"], "price_chf"].dropna()
    d, ci = boot_diff_ci(hi, lo)
    print(f"{resort:10} CHF {d:+6.2f}  95% CI [{ci[0]:+6.2f}, {ci[1]:+6.2f}]  "
          f"({d / lo.mean() * 100:+5.1f}%)   n={len(hi)}/{len(lo)}")

print("\nmean price by resort × holiday × day type:")
print(day.pivot_table(index=["resort", "holiday"], columns="is_weekend",
                      values="price_chf", aggfunc="mean", dropna=False).round(2))


=== weekend premium ===
Laax       CHF  +3.50  95% CI [ -1.16,  +7.98]  ( +4.8%)   n=40/90
Titlis     CHF  +4.37  95% CI [ +2.89,  +5.76]  ( +7.7%)   n=43/109

=== holiday premium ===
Laax       CHF  +8.09  95% CI [ +3.44, +12.50]  (+11.4%)   n=49/81
Titlis     CHF  +3.08  95% CI [ +1.63,  +4.54]  ( +5.4%)   n=42/110

mean price by resort and holiday window:
resort      Laax  Titlis
holiday                 
christmas  82.53   61.53
easter     64.18   56.85
sport      91.78   64.06
NaN        71.09   57.10
=== holiday premium, WEEKDAYS ONLY ===
Laax       CHF  +9.44  95% CI [ +3.53, +15.42]  (+13.5%)   n=31/59
Titlis     CHF  +3.33  95% CI [ +1.54,  +5.11]  ( +6.0%)   n=26/83

=== holiday premium, WEEKENDS ONLY ===
Laax       CHF  +4.53  95% CI [ -2.64, +11.95]  ( +6.1%)   n=18/22
Titlis     CHF  +0.97  95% CI [ -1.36,  +3.34]  ( +1.6%)   n=16/27

mean price by resort × holiday × day type:
is_weekend        False  True 
resort holiday                
Laax   christmas  84.75  78.83
    

# Project B: Feasibility Testing
Testing the feasibility of Project B - what people pay for and what they play on Steam